In [5]:
import pandas as pd
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_DIR = Path.cwd()
BOOKS_CSV = DATA_DIR / "Books.csv"
RATINGS_CSV = DATA_DIR / "Ratings.csv"
USERS_CSV = DATA_DIR / "Users.csv"

DB_PATH = DATA_DIR / "books.db"

books = pd.read_csv(BOOKS_CSV, low_memory=False)
ratings = pd.read_csv(RATINGS_CSV, low_memory=False)
users = pd.read_csv(USERS_CSV, low_memory=False)





In [6]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [7]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [8]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [9]:
users["Age"] = pd.to_numeric(users['Age'], errors='coerce')

users.loc[(users["Age"] < 5) | (users['Age'] > 100), 'Age'] = pd.NA

for col in ["Location"]:
    if col in users.columns:
        users[col] = users[col].astype(str).str.strip().replace({'': pd.NA, 'NaN': pd.NA})

if 'User-ID' in users.columns:
    users = users.drop_duplicates(subset=["User-ID"])
else:
    users = users.drop_duplicates()

users.head(), users["Age"].describe()

(   User-ID                            Location   Age
 0        1                  nyc, new york, usa   NaN
 1        2           stockton, california, usa  18.0
 2        3     moscow, yukon territory, russia   NaN
 3        4           porto, v.n.gaia, portugal  17.0
 4        5  farnborough, hants, united kingdom   NaN,
 count    166848.000000
 mean         34.746638
 std          13.633051
 min           5.000000
 25%          24.000000
 50%          32.000000
 75%          44.000000
 max         100.000000
 Name: Age, dtype: float64)

In [10]:
users['Age'].isna().sum(), len(users)

users.loc[users['Age'].isna()].head(10)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
2,3,"moscow, yukon territory, russia",NaN
4,5,"farnborough, hants, united kingdom",NaN
6,7,"washington, dc, usa",NaN
7,8,"timmins, ontario, canada",NaN
8,9,"germantown, tennessee, usa",NaN
11,12,"fort bragg, california, usa",NaN
13,14,"mediapolis, iowa, usa",NaN
14,15,"calgary, alberta, canada",NaN
15,16,"albuquerque, new mexico, usa",NaN


In [11]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [17]:
ratings['Book-Rating'] = pd.to_numeric(ratings['Book-Rating'], errors="coerce")
ratings = ratings[ratings['Book-Rating'].notna()]
ratings["Book-Rating"] = ratings["Book-Rating"].clip(lower=0, upper=10).astype("Int16")

for key in ['User-ID', 'ISBN']:
    if key in ratings.columns:
        ratings = ratings[ratings[key].notna() & (ratings[key].astype(str).str.strip() != '')]

ratings["ISBN"] = ratings["ISBN"].astype(str).str.strip()

if set(['User-ID','ISBN']).issubset(ratings.columns):
    ratings = ratings.drop_duplicates(subset=['User-ID', 'ISBN', 'Book-Rating'])
else:
    ratings = ratings.drop_duplicates()

ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
